# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yara-08/ML-FlyRank-Internship2/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

> ### Finding 1 — The Content Performance Curve

> The paper reports that content reaches its highest average health score at 61–90 days, declines after 270 days, and shows a higher health score again among some 365+ day content. The paper also narrows this interpretation by noting that the older-page rebound is concentrated among pages that were refreshed.

> **Methodology question:** What exactly defines the performance outcome for each age cohort, and does the cohort comparison account for other differences between pages that may be related to both content age and performance? For example, content age is identified in the methodology as a potential confounding variable in model-performance comparisons. I would want to understand whether the observed age pattern is best interpreted as a descriptive association within this portfolio rather than evidence that age itself causes performance to decline. This would help ensure that the claim stays aligned with the observational design.

> ### Finding 2 — The Age-Freshness Matrix

> The paper reports that 365+ day content refreshed within the previous 30 days has a health score of 44.62, close to the 44.12 score for young, fresh content. It presents this as evidence that older content can perform nearly as well as new content when refreshed. The paper also acknowledges survivor bias in the small 365+ × 361+ cell.

> **Methodology question:** How is the refreshed-versus-unrefreshed comparison constructed, and does the validation design support interpreting the difference as an effect of refreshing rather than an association? Pages selected for refresh may differ systematically from pages that are not refreshed, such as having stronger historical demand or receiving more attention. A comparison using matched pages or a before-and-after design with an appropriate comparison group could strengthen the evidence for a refresh effect. For the current analysis, I would keep the finding framed as an observed association within this portfolio rather than a causal estimate.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*



> My Week-5 model already used a grouped train/test split by `client_id`, so I am treating that grouped result as the primary validation estimate rather than retroactively describing Week 5 as a random split. For this audit, I additionally ran the same Random Forest under a random 80/20 split to measure how much the validation result changes when client-level separation is removed.

> The random split produced a higher measured F1 score (0.877) than the grouped-by-client split (0.831). The random split had 31 clients represented in both the training and test sets, while the grouped split had no shared clients. This difference suggests that allowing content from the same clients into both sets makes the validation task easier.

>For the question of performance on clients not seen during training, the grouped-by-client result is the more conservative and relevant estimate. The observed gap between the two validation designs is itself a useful finding: the measured model performance depends on the split design. These results support using the grouped result for decision-support claims rather than presenting the higher random-split score as general performance.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

!git clone https://github.com/yara-08/ML-FlyRank-Internship2.git
import pandas as pd
# Load dataset
df = pd.read_csv("/content/ML-FlyRank-Internship2/data/raw/content_refresh_anonymized.csv")

# Section 2 — Random vs grouped validation
# The Week-5 grouped split is the primary honest evaluation.
# The random split is included as a diagnostic comparison.

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# --------------------------------------------------
# Target
# --------------------------------------------------

df["target"] = (df["trend_direction"] == "down").astype(int)

# --------------------------------------------------
# Feature exclusions
# --------------------------------------------------

drop_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "target",
]

# Exclude Week-4 decision outputs if they are present.
# They are not used as model features.
for col in ["baseline_score", "baseline_prediction"]:
    if col in df.columns:
        drop_columns.append(col)


# --------------------------------------------------
# Shared preprocessing function
# --------------------------------------------------

def prepare_features(train_df, test_df):

    X_train = train_df.drop(columns=drop_columns).copy()
    X_test = test_df.drop(columns=drop_columns).copy()

    y_train = train_df["target"]
    y_test = test_df["target"]

    # Encode categorical variables
    X_train = pd.get_dummies(X_train, drop_first=True)
    X_test = pd.get_dummies(X_test, drop_first=True)

    # Match train/test feature columns
    X_train, X_test = X_train.align(
        X_test,
        join="left",
        axis=1,
        fill_value=0,
    )

    # Missingness indicators
    missing_cols = [
        "char_count",
        "word_count",
        "competition",
        "search_volume",
        "cpc",
        "scroll_rate",
    ]

    for col in missing_cols:
        X_train[f"{col}_missing"] = X_train[col].isna().astype(int)
        X_test[f"{col}_missing"] = X_test[col].isna().astype(int)

        # Use training-set median only
        median = X_train[col].median()

        X_train[col] = X_train[col].fillna(median)
        X_test[col] = X_test[col].fillna(median)

    return X_train, X_test, y_train, y_test


# --------------------------------------------------
# Evaluation function
# --------------------------------------------------

def evaluate_random_forest(train_df, test_df):

    X_train, X_test, y_train, y_test = prepare_features(
        train_df,
        test_df,
    )

    rf = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
    )

    rf.fit(X_train, y_train)

    predictions = rf.predict(X_test)

    return {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1 Score": f1_score(y_test, predictions),
        "Positive rate": y_test.mean(),
    }


# --------------------------------------------------
# 1. Random split
# --------------------------------------------------

random_train, random_test = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["target"],
)

random_metrics = evaluate_random_forest(
    random_train,
    random_test,
)


# --------------------------------------------------
# 2. Grouped split by client
# --------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

group_train_idx, group_test_idx = next(
    gss.split(
        df,
        groups=df["client_id"],
    )
)

group_train = df.iloc[group_train_idx].copy()
group_test = df.iloc[group_test_idx].copy()

group_metrics = evaluate_random_forest(
    group_train,
    group_test,
)


# --------------------------------------------------
# Validation comparison
# --------------------------------------------------

comparison = pd.DataFrame(
    {
        "Validation design": [
            "Random split",
            "Grouped by client",
        ],
        "Accuracy": [
            random_metrics["Accuracy"],
            group_metrics["Accuracy"],
        ],
        "Precision": [
            random_metrics["Precision"],
            group_metrics["Precision"],
        ],
        "Recall": [
            random_metrics["Recall"],
            group_metrics["Recall"],
        ],
        "F1 Score": [
            random_metrics["F1 Score"],
            group_metrics["F1 Score"],
        ],
        "Positive rate": [
            random_metrics["Positive rate"],
            group_metrics["Positive rate"],
        ],
    }
).round(3)

display(comparison)


# --------------------------------------------------
# Group separation check
# --------------------------------------------------

shared_clients = (
    set(group_train["client_id"])
    & set(group_test["client_id"])
)

print("Random split:")
print(f"  Training rows: {len(random_train):,}")
print(f"  Testing rows: {len(random_test):,}")
print(f"  Clients in training: {random_train['client_id'].nunique()}")
print(f"  Clients in testing: {random_test['client_id'].nunique()}")
print(
    f"  Shared clients: "
    f"{len(set(random_train['client_id']) & set(random_test['client_id']))}"
)

print("\nGrouped split:")
print(f"  Training rows: {len(group_train):,}")
print(f"  Testing rows: {len(group_test):,}")
print(f"  Clients in training: {group_train['client_id'].nunique()}")
print(f"  Clients in testing: {group_test['client_id'].nunique()}")
print(f"  Shared clients: {len(shared_clients)}")

print("\nF1 difference:")
print(
    f"  Random - Grouped = "
    f"{random_metrics['F1 Score'] - group_metrics['F1 Score']:.3f}"
)

fatal: destination path 'ML-FlyRank-Internship2' already exists and is not an empty directory.


,Validation design,Accuracy,Precision,Recall,F1 Score,Positive rate
0,Random split,0.863,0.851,0.905,0.877,0.542
1,Grouped by client,0.826,0.824,0.838,0.831,0.511


Random split:
  Training rows: 24,000
  Testing rows: 6,000
  Clients in training: 32
  Clients in testing: 31
  Shared clients: 31

Grouped split:
  Training rows: 23,837
  Testing rows: 6,163
  Clients in training: 25
  Clients in testing: 7
  Shared clients: 0

F1 difference:
  Random - Grouped = 0.047


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*



> I audited the final feature set against three leakage risks: label-derived features, future or overlapping windows, and decision-derived features.

> First, the target is defined from `trend_direction`, with `trend_pct` also identified as a label-derived field in the data documentation. I excluded `trend_direction`, `trend_pct`, and the derived `target` column from the model features. I also excluded `content_id` and `client_id`; these are identifiers used for grouping or splitting rather than predictive inputs.

> Second, I checked for decision-derived features. The Week-4 `baseline_score` and `baseline_prediction` were excluded from the Random Forest features because they encode an existing decision rule. They remain useful for comparison but should not become model inputs.

> Third, I considered temporal or overlapping-window leakage. The dataset contains trailing performance features such as `impressions_90d`, `impressions_last_30d`, and `impressions_prev_30d`. The starter data documentation does not provide a separate prediction timestamp or the exact time window used to calculate `trend_pct`, so I cannot prove from this CSV alone that every performance window is strictly before the label window. I therefore treat temporal separation as a limitation rather than claiming that it has been completely verified.

> I also inspected real errors from the grouped-by-client evaluation. The model produced 563 false positives and 511 false negatives, for 1,074 errors out of 6,163 held-out pages. The false-positive examples included stable pages with weak recent activity, such as very few sessions or clicks, which can resemble a declining page. The false-negative examples included declining pages that still had substantial historical impressions, showing that strong historical visibility can coexist with a measured downward trend.

> These examples show that the model can encounter mixed performance signals and should not be interpreted as a definitive classification of which pages require a refresh. The measured result is better treated as directional decision-support.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 3 — Leakage audit

# -----------------------------------------
# 1. Define columns that must never be features
# -----------------------------------------

label_columns = [
    "trend_direction",
    "trend_pct",
    "target",
]

identifier_columns = [
    "content_id",
    "client_id",
]

decision_derived_columns = [
    "baseline_score",
    "baseline_prediction",
]

excluded_columns = (
    label_columns
    + identifier_columns
    + decision_derived_columns
)

# Only use columns that exist
excluded_columns = [
    col for col in excluded_columns
    if col in df.columns
]

# This now represents the actual pre-encoding feature set
feature_columns = [
    col for col in df.columns
    if col not in excluded_columns
]


# -----------------------------------------
# 2. Check label-derived leakage
# -----------------------------------------

label_present = [
    col for col in label_columns
    if col in feature_columns
]

print("Label-derived columns included as features:")
print(label_present)

assert len(label_present) == 0, (
    "Potential label-derived leakage detected."
)

print("PASS: No label-derived columns are in the feature set.")


# -----------------------------------------
# 3. Check identifier leakage
# -----------------------------------------

ids_present = [
    col for col in identifier_columns
    if col in feature_columns
]

print("\nIdentifier columns included as features:")
print(ids_present)

assert len(ids_present) == 0, (
    "Identifier leakage detected."
)

print("PASS: IDs are excluded from model features.")


# -----------------------------------------
# 4. Check decision-derived leakage
# -----------------------------------------

decision_present = [
    col for col in decision_derived_columns
    if col in feature_columns
]

print("\nDecision-derived columns included as features:")
print(decision_present)

assert len(decision_present) == 0, (
    "Decision-derived feature detected."
)

print("PASS: Week-4 decision outputs are excluded.")


# -----------------------------------------
# 5. Show actual final feature set
# -----------------------------------------

print(f"\nFinal pre-encoding feature count: {len(feature_columns)}")

print("\nFinal pre-encoding feature set:")
print(feature_columns)


# -----------------------------------------
# 6. Flag temporal-window features
# -----------------------------------------

temporal_keywords = [
    "90d",
    "last_30d",
    "prev_30d",
]

temporal_features = [
    col
    for col in feature_columns
    if any(keyword in col for keyword in temporal_keywords)
]

print("\nTemporal-window features requiring timing review:")

for col in temporal_features:
    print("-", col)


# -----------------------------------------
# 7. Leakage audit summary
# -----------------------------------------

audit = pd.DataFrame({
    "Leakage check": [
        "Label-derived columns",
        "Identifier columns",
        "Decision-derived columns",
        "Temporal-window features",
    ],
    "Result": [
        "PASS",
        "PASS",
        "PASS",
        "REVIEW LIMITATION",
    ],
    "Evidence": [
        "trend_direction, trend_pct, and target excluded",
        "content_id and client_id excluded",
        "baseline_score and baseline_prediction excluded",
        "Exact trend_pct timing is not provided in the starter CSV",
    ],
})

display(audit)



# -----------------------------------------
# 8. Real failure examples
# -----------------------------------------

# Reuse the grouped split from Section 2
# and train the same Random Forest configuration.

X_train, X_test, y_train, y_test = prepare_features(
    group_train,
    group_test
)

rf_grouped = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
)

rf_grouped.fit(X_train, y_train)

group_predictions = rf_grouped.predict(X_test)

# Attach predictions to the original held-out rows
results = group_test.copy()

results["Actual"] = y_test.values
results["Predicted"] = group_predictions

results["Error type"] = "Correct"

results.loc[
    (results["Actual"] == 0) &
    (results["Predicted"] == 1),
    "Error type"
] = "False Positive"

results.loc[
    (results["Actual"] == 1) &
    (results["Predicted"] == 0),
    "Error type"
] = "False Negative"


# -----------------------------------------
# Error counts
# -----------------------------------------

error_counts = (
    results["Error type"]
    .value_counts()
    .reindex(
        ["False Positive", "False Negative", "Correct"],
        fill_value=0
    )
)

print("Grouped-model outcome counts:")
display(error_counts.to_frame("Rows"))


# -----------------------------------------
# Select real failure examples
# -----------------------------------------

error_columns = [
    "Error type",
    "trend_direction",
    "impressions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_age_days",
]

# Keep only columns that exist
error_columns = [
    col for col in error_columns
    if col in results.columns
]


# Three false positives
false_positives = results[
    results["Error type"] == "False Positive"
][error_columns].head(3)

print("\nThree false-positive examples:")
display(false_positives)


# Three false negatives
false_negatives = results[
    results["Error type"] == "False Negative"
][error_columns].head(3)

print("\nThree false-negative examples:")
display(false_negatives)

Label-derived columns included as features:
[]
PASS: No label-derived columns are in the feature set.

Identifier columns included as features:
[]
PASS: IDs are excluded from model features.

Decision-derived columns included as features:
[]
PASS: Week-4 decision outputs are excluded.

Final pre-encoding feature count: 40

Final pre-encoding feature set:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 

,Leakage check,Result,Evidence
0,Label-derived columns,PASS,"trend_direction, trend_pct, and target excluded"
1,Identifier columns,PASS,content_id and client_id excluded
2,Decision-derived columns,PASS,baseline_score and baseline_prediction excluded
3,Temporal-window features,REVIEW LIMITATION,Exact trend_pct timing is not provided in the ...


Grouped-model outcome counts:


,Rows
Error type,
False Positive,563
False Negative,511
Correct,5089



Three false-positive examples:


,Error type,trend_direction,impressions_90d,impressions_last_30d,impressions_prev_30d,clicks_last_30d,sessions_last_30d,ctr,avg_position,days_since_last_update,content_age_days
13,False Positive,stable,307,85,77,0,3,0.00,39.8,103,238
26,False Positive,stable,2426,743,818,0,5,0.12,30.0,13,300
36,False Positive,stable,371,77,95,1,1,1.35,5.4,20,187



Three false-negative examples:


,Error type,trend_direction,impressions_90d,impressions_last_30d,impressions_prev_30d,clicks_last_30d,sessions_last_30d,ctr,avg_position,days_since_last_update,content_age_days
1,False Negative,down,15320,2501,5915,2,3,0.05,20.3,25,445
81,False Negative,down,320,84,106,0,0,0.31,9.6,22,502
129,False Negative,down,2159,403,663,0,1,0.05,21.3,7,421


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*



> My strongest Week-5 claim was that the Random Forest "improved the model's ability to identify declining pages." After the validation and leakage audit, I would make this claim more precise.

> **Original claim:**
The Random Forest achieved higher accuracy, precision, recall, and F1 than the Week-4 baseline, suggesting that using a broader set of features improved the model's ability to identify declining pages.

> **Rewritten claim:**
On the held-out grouped-by-client test set, the Random Forest measured an F1 score of 0.831, compared with 0.052 for the Week-4 baseline. The model therefore showed better measured classification performance on this dataset and split. The result is directional evidence that the Random Forest features were useful for this classification task, but it does not establish causation or prove that the model will generalize to other clients or datasets.

> The model should be used as **decision-support** for identifying pages with signals associated with the observed declining label, rather than as a definitive rule for deciding which pages require a refresh. The observed errors and the unresolved temporal-window limitation should also be considered when interpreting the result.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 4 — Claim rewrite
# Verify the measured values used in the rewritten claim.

claim_metrics = pd.DataFrame({
    "Method": [
        "Week 4 Baseline",
        "Random Forest",
    ],
    "Validation": [
        "Grouped by client",
        "Grouped by client",
    ],
    "F1 Score": [
        0.052,
        0.831,
    ],
    "Accuracy": [
        0.482,
        0.826,
    ],
    "Precision": [
        0.399,
        0.824,
    ],
    "Recall": [
        0.028,
        0.838,
    ],
})

display(claim_metrics)

print(
    "Measured F1 difference:",
    round(
        claim_metrics.loc[1, "F1 Score"]
        - claim_metrics.loc[0, "F1 Score"],
        3
    )
)

print("\nSafe interpretation:")
print(
    "The Random Forest showed better measured classification "
    "performance than the Week-4 baseline on the grouped test set."
)
print(
    "This is directional decision-support evidence, not causal proof "
    "or a guarantee of generalization."
)

,Method,Validation,F1 Score,Accuracy,Precision,Recall
0,Week 4 Baseline,Grouped by client,0.052,0.482,0.399,0.028
1,Random Forest,Grouped by client,0.831,0.826,0.824,0.838


Measured F1 difference: 0.779

Safe interpretation:
The Random Forest showed better measured classification performance than the Week-4 baseline on the grouped test set.
This is directional decision-support evidence, not causal proof or a guarantee of generalization.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.